# DLR Carrier Board — Power & Signal Integrity Analysis

Single-PCB CM4 carrier for an outdoor solar-powered RTU. Soldered Quectel BG770A (LTE Cat-M1, NA bands), FLIR Lepton 3.5 thermal imager (SPI), DHT22 + SI1145 + ADS1115/YL-83 sensors, MPPT charger from 20W solar panel, LiFePO4 4S battery + BMS, buck for 5V SoM rail, AP2112K LDO for 3.3V analog.

This notebook derives the buck input current envelope, daily energy budget, I2C pull-up sizing, SPI critical trace length, and USB 2.0 differential pair geometry.

## Assumptions
- LiFePO4 4S battery: V_min=10.0V (2.5V/cell cutoff), V_nom=12.8V, V_max=14.6V (3.65V/cell charge top)
- 4 Ah battery cell — 50 Wh nominal, 90% usable depth-of-discharge per LiFePO4 chemistry tolerance
- 20W solar panel rated at STC; winter NE US worst case = 2.5 sun-hours/day, annual avg 4.0
- Generic synchronous buck regulator, η=0.90±0.03 across the 10–14.6V Vin range
- AP2112K LDO @ 3.3V — datasheet dropout 250 mV at 600 mA (we run ~14 mA peak)
- CM4 always-on idle baseline: 120 mA at 5V (Pi OS Lite, headless, no WiFi/BT)
- Duty cycle: sensor wake 5s every 60s, cellular TX 5s every 15 min; CM4 in idle the rest of the time
- I2C bus cap: 4 device slots × 10pF + ~30pF traces + 5pF connector = 75pF estimate
- FR4 substrate εr=4.3; v_prop = c/√εr ≈ 1.45×10⁸ m/s
- 4-layer 1.6mm board, F.Cu to In1.GND prepreg height = 0.20 mm (controlled-impedance fab order)
- Parasitic inductance and trace resistance neglected for power rails (short runs, bulk caps present)

In [1]:
"""Universal constants for DLR carrier theory derivation."""

import math
from typing import Final

import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

# === Battery: LiFePO4 4S ===
# Reason: 3.2V/cell nominal x 4 = 12.8V; 2.5V cutoff x 4 = 10.0V; 3.65V charge x 4 = 14.6V
V_BAT_NOM: Final = 12.8 * ureg.V
V_BAT_MIN: Final = 10.0 * ureg.V
V_BAT_MAX: Final = 14.6 * ureg.V
BAT_CAPACITY: Final = 4.0 * ureg.A * ureg.hour
BAT_USABLE_FRACTION: Final = 0.90

# === Buck (5V rail) and LDO (3V3 rail) ===
V_RAIL_5V: Final = 5.0 * ureg.V
V_RAIL_3V3: Final = 3.3 * ureg.V
BUCK_EFFICIENCY: Final = ufloat(0.90, 0.03) * ureg.dimensionless  # generic synchronous buck
LDO_DROPOUT_AT_600MA: Final = 250 * ureg.mV  # AP2112K datasheet

# === Solar / power budget ===
PANEL_RATING: Final = 20 * ureg.W
SUN_HOURS_WORST: Final = 2.5 * ureg.hour  # December NE US
SUN_HOURS_AVG: Final = 4.0 * ureg.hour    # Annual avg NE US
MPPT_EFFICIENCY: Final = 0.90
I_5V_PEAK: Final = 3920 * ureg.mA
I_5V_TYPICAL: Final = 1660 * ureg.mA

# === I2C bus (UM10204 Rev 7.0 Table 10, fast mode 400 kHz) ===
V_OL_MAX: Final = 0.4 * ureg.V
I_OL_FAST: Final = 3.0 * ureg.mA
T_RISE_FAST_MAX: Final = 300 * ureg.ns
C_BUS_ESTIMATED: Final = 75 * ureg.pF

# === FLIR Lepton 3.5 SPI ===
F_SPI_MAX: Final = 20 * ureg.MHz
BW_HARMONIC_FACTOR: Final = 5  # 5x f_clk for clean edges

# === FR4 substrate + USB 2.0 ===
EPSILON_R_FR4: Final = 4.3
V_PROP_FR4: Final = 3e8 / math.sqrt(EPSILON_R_FR4) * ureg.m / ureg.s
Z_DIFF_USB_TARGET: Final = 90 * ureg.ohm
USB_SKEW_MAX: Final = 100 * ureg.ps
H_TO_GND: Final = 0.20 * ureg.mm  # F.Cu to In1.GND on 4L 1.6mm board
W_TRACE_USB: Final = 0.20 * ureg.mm  # 8 mil
G_GAP_USB: Final = 0.15 * ureg.mm    # 6 mil

print("Constants loaded.")

Constants loaded.


## Section 1: Power Architecture

Buck input current via power conservation. Worst case is at minimum battery voltage (post-cutoff approach):

$$ I_{buck,in} = \frac{V_{out} \cdot I_{out}}{V_{in} \cdot \eta} $$

LDO V_in headroom must clear (V_out + V_dropout) at peak load:

$$ \Delta V_{headroom} = V_{rail,5V} - V_{rail,3V3} - V_{dropout} $$

In [2]:
# Buck input current envelope
i_in_peak_worst = (V_RAIL_5V * I_5V_PEAK) / (V_BAT_MIN * BUCK_EFFICIENCY)
i_in_typ_nom = (V_RAIL_5V * I_5V_TYPICAL) / (V_BAT_NOM * BUCK_EFFICIENCY)
i_bat_1c = BAT_CAPACITY / (1 * ureg.hour)

print(f"I_buck_in (peak @ V_BAT_MIN): {i_in_peak_worst.to(ureg.mA):.0f}")
print(f"I_buck_in (typ @ V_BAT_NOM): {i_in_typ_nom.to(ureg.mA):.0f}")
print(f"Battery 1C discharge limit: {i_bat_1c.to(ureg.A):.1f}")

# LDO headroom check
v_ldo_headroom = V_RAIL_5V - V_RAIL_3V3 - LDO_DROPOUT_AT_600MA
print(f"\nLDO V_in headroom: {v_ldo_headroom.to(ureg.V):.2f}")

I_buck_in (peak @ V_BAT_MIN): 2178+/-73 milliampere
I_buck_in (typ @ V_BAT_NOM): 720+/-24 milliampere
Battery 1C discharge limit: 4.0 ampere

LDO V_in headroom: 1.45 volt


## Section 2: Daily Energy Budget

Realistic operating profile — CM4 always-on at idle, brief active wakes for sampling, cellular bursts every 15 min. Average current is dominated by CM4 idle.

$$ I_{avg} = I_{idle} + \sum_i (I_i - I_{idle}) \cdot D_i $$

where $D_i = t_{on,i} / t_{period,i}$ is the duty cycle of each consumer.

In [3]:
# Duty cycle constants
SAMPLE_INTERVAL: Final = 1 * ureg.minute
SAMPLE_DURATION: Final = 5 * ureg.s
TX_INTERVAL: Final = 15 * ureg.minute
TX_DURATION: Final = 5 * ureg.s

# Per-component currents at 5V
I_CM4_IDLE: Final = 120 * ureg.mA
I_CM4_ACTIVE: Final = 1400 * ureg.mA
I_LEPTON_ON: Final = 150 * ureg.mA
I_BG770_PSM: Final = 1 * ureg.mA
I_BG770_TX: Final = 250 * ureg.mA
I_SENSORS_ACTIVE: Final = 9 * ureg.mA

sample_duty = (SAMPLE_DURATION / SAMPLE_INTERVAL).to(ureg.dimensionless)
tx_duty = (TX_DURATION / TX_INTERVAL).to(ureg.dimensionless)

i_avg = (
    I_CM4_IDLE
    + (I_CM4_ACTIVE - I_CM4_IDLE) * sample_duty
    + I_LEPTON_ON * sample_duty
    + I_BG770_PSM
    + (I_BG770_TX - I_BG770_PSM) * tx_duty
    + I_SENSORS_ACTIVE * sample_duty
)
p_avg = i_avg * V_RAIL_5V
e_daily = p_avg * (24 * ureg.hour)

print(f"Sample duty: {sample_duty.magnitude:.4f}")
print(f"TX duty: {tx_duty.magnitude:.4f}")
print(f"Average current: {i_avg.to(ureg.mA):.0f}")
print(f"Average power: {p_avg.to(ureg.W):.2f}")
print(f"Daily energy: {e_daily.to(ureg.W * ureg.hour):.1f}")

# Solar production
e_pv_winter = PANEL_RATING * SUN_HOURS_WORST * MPPT_EFFICIENCY
e_pv_avg = PANEL_RATING * SUN_HOURS_AVG * MPPT_EFFICIENCY
winter_margin = (e_pv_winter / e_daily).to(ureg.dimensionless)

print(f"\nPV winter (worst): {e_pv_winter.to(ureg.W * ureg.hour):.1f}")
print(f"PV annual avg: {e_pv_avg.to(ureg.W * ureg.hour):.1f}")
print(f"Winter margin: {winter_margin.magnitude:.2f}x")

# Battery autonomy at zero PV
# Reason: V_BAT * BAT_CAPACITY (A*hr) gives Wh directly; no extra divide
e_bat_usable = V_BAT_NOM * BAT_CAPACITY * BAT_USABLE_FRACTION
days_autonomy = (e_bat_usable / e_daily).to(ureg.dimensionless)
print(f"\nBattery usable energy: {e_bat_usable.to(ureg.W * ureg.hour):.1f}")
print(f"Autonomy at zero PV: {days_autonomy.magnitude:.2f} days")

Sample duty: 0.0833
TX duty: 0.0056
Average current: 242 milliampere
Average power: 1.21 watt
Daily energy: 29.1 hour * watt

PV winter (worst): 45.0 hour * watt
PV annual avg: 72.0 hour * watt
Winter margin: 1.55x

Battery usable energy: 46.1 hour * watt
Autonomy at zero PV: 1.58 days


## Section 3: I2C Pull-Up Sizing (400 kHz, 3.3V)

Per UM10204 Rev 7.0 Table 10, fast mode requires R within bounds set by sink current and rise time:

$$ R_{min} = \frac{V_{cc} - V_{OL,max}}{I_{OL}} \qquad R_{max} = \frac{t_{rise,max}}{0.8473 \cdot C_{bus}} $$

The 0.8473 factor is the natural log ratio for RC charging from 30% to 70% of V_cc.

In [4]:
r_pullup_min = (V_RAIL_3V3 - V_OL_MAX) / I_OL_FAST
r_pullup_max = T_RISE_FAST_MAX / (0.8473 * C_BUS_ESTIMATED)

R_PULLUP_SELECTED: Final = 2.2 * ureg.kohm  # E24 standard, in range
t_rise_actual = 0.8473 * R_PULLUP_SELECTED * C_BUS_ESTIMATED

print(f"R_min: {r_pullup_min.to(ureg.ohm):.0f}")
print(f"R_max: {r_pullup_max.to(ureg.ohm):.0f}")
print(f"R_selected (E24): {R_PULLUP_SELECTED}")
print(f"In range: {r_pullup_min < R_PULLUP_SELECTED < r_pullup_max}")
print(f"Rise time at selected R: {t_rise_actual.to(ureg.ns):.0f} (max {T_RISE_FAST_MAX})")

R_min: 967 ohm
R_max: 4721 ohm
R_selected (E24): 2.2 kiloohm
In range: True
Rise time at selected R: 140 nanosecond (max 300 nanosecond)


## Section 4: SPI Signal Integrity (FLIR Lepton, 20 MHz)

Lepton 3.5 SPI runs at 20 MHz. Edge rise time required for clean edges (5x harmonic content):

$$ t_r \le \frac{0.35}{5 \cdot f_{clk}} $$

Critical trace length above which transmission-line behavior matters:

$$ L_{crit} = \frac{t_r \cdot v_{prop}}{2} $$

Design rule: keep traces below $L_{crit}/5$ to be safely lumped.

In [5]:
bw_required = BW_HARMONIC_FACTOR * F_SPI_MAX
t_rise_spi = 0.35 / bw_required
l_crit_spi = (t_rise_spi * V_PROP_FR4 / 2).to(ureg.mm)
l_design_rule = (l_crit_spi / 5).to(ureg.mm)

print(f"Required BW (5x f_clk): {bw_required.to(ureg.MHz):.0f}")
print(f"Edge rise time spec: {t_rise_spi.to(ureg.ns):.2f}")
print(f"Critical TL length: {l_crit_spi:.0f}")
print(f"Design rule (lumped trace budget): < {l_design_rule:.0f}")

Required BW (5x f_clk): 100 megahertz
Edge rise time spec: 3.50 nanosecond
Critical TL length: 253 millimeter
Design rule (lumped trace budget): < 51 millimeter


## Section 5: USB 2.0 Differential Routing (BG770A)

USB 2.0 High-Speed requires Z_diff = 90Ω ± 10% on the D+/D- pair, with intra-pair skew under 100 ps.

Single-ended microstrip impedance (Wadell):

$$ Z_0 = \frac{60}{\sqrt{\varepsilon_r}} \ln\left(\frac{8h}{w} + \frac{w}{4h}\right) $$

Edge-coupled differential approximation:

$$ Z_{diff} \approx 2 \cdot Z_0 \cdot \left(1 - 0.48 \cdot e^{-0.96 \cdot s/h}\right) $$

where $s$ is the gap between traces and $h$ is the height to the GND reference plane.

In [6]:
# Strip pint units for Wadell formulas (closed-form takes plain ratios)
h = H_TO_GND.to(ureg.mm).magnitude
w = W_TRACE_USB.to(ureg.mm).magnitude
s = G_GAP_USB.to(ureg.mm).magnitude

z0_se = (60 / math.sqrt(EPSILON_R_FR4)) * math.log(8 * h / w + w / (4 * h)) * ureg.ohm
z_diff = 2 * z0_se * (1 - 0.48 * math.exp(-0.96 * s / h))

print(f"Stack-up: w={w} mm, gap={s} mm, h={h} mm to GND")
print(f"Z_0 (single-ended): {z0_se:.1f}")
print(f"Z_diff: {z_diff:.1f} (target {Z_DIFF_USB_TARGET} +/-10%)")
tol_low = Z_DIFF_USB_TARGET * 0.9
tol_high = Z_DIFF_USB_TARGET * 1.1
print(f"In tolerance ({tol_low:.0f}-{tol_high:.0f}): {tol_low < z_diff < tol_high}")

# Skew length budget
skew_length = (USB_SKEW_MAX * V_PROP_FR4).to(ureg.mm)
print(f"\nMax intra-pair skew length: {skew_length:.1f} ({USB_SKEW_MAX})")

Stack-up: w=0.2 mm, gap=0.15 mm, h=0.2 mm to GND
Z_0 (single-ended): 61.1 ohm
Z_diff: 93.6 ohm (target 90 ohm +/-10%)
In tolerance (81 ohm-99 ohm): True

Max intra-pair skew length: 14.5 millimeter (100 picosecond)


## Expected Values

| Parameter | Value | Tolerance | Source |
|-----------|-------|-----------|--------|
| Buck input current at peak (V_BAT_MIN) | 2.18 A | +/-0.07 A | Power conservation, η=0.90+/-0.03 |
| Buck input current typical (V_BAT_NOM) | 0.72 A | +/-0.02 A | Power conservation |
| Battery 1C limit (4 Ah cell) | 4.0 A | — | Datasheet |
| LDO V_in headroom at 3V3 | 1.45 V | — | AP2112K dropout 250 mV at 600 mA |
| Daily average current (5V) | 242 mA | — | Realistic duty cycle |
| Daily energy budget | 29.0 Wh | — | I_avg × 5V × 24h |
| PV production (winter worst) | 45.0 Wh/day | — | 20W × 2.5 h × 0.90 MPPT η |
| Winter margin | 1.55x | — | E_pv / E_daily |
| Battery autonomy (50 Wh, 90% DoD) | 1.59 days | — | E_bat_usable / E_daily |
| I2C pull-up | 2.2 kΩ | E24 | UM10204 Table 10, in 967Ω–4.72kΩ range |
| I2C rise time at selected R | 140 ns | — | < 300 ns spec, 53% margin |
| SPI critical trace length | 254 mm | — | TL threshold; design rule keep < 51 mm |
| USB Z_diff | 93.6 Ω | target 90 +/-9 Ω | Wadell, 8/6 mil on h=0.2mm |
| USB skew length budget | 14.5 mm | — | 100 ps × v_prop |

**Design notes:**

- **Battery sizing:** 50 Wh autonomy is 1.59 days at zero PV — winter cloud weeks could deplete. Recommend stepping to **100 Wh battery** for ~3 days autonomy.
- **Buck topology:** must accept V_in = 10.0 V to 14.6 V continuously (1.46x range). Synchronous buck with low-Rds_on FETs preferred for the 0.90 efficiency assumption.
- **USB diff impedance** assumes a controlled-impedance fab order with the specific 4-layer stack (h=0.20mm to GND). Generic 1.6mm fab will not hit tolerance.
- **I2C bus capacitance** estimate (75 pF) covers 4 device slots — verify after layout, re-derive if exceeded.